## Step-3.1 — Accuracy-Only Training Loop

### Environment & Imports

This notebook implements accuracy-only baseline training for CIFAR-10 under a fixed, pre-committed experimental configuration.  
All hyperparameters and design choices were locked prior to execution (Step-3.0).

This section verifies the execution environment and imports required libraries.


In [1]:
# Standard libraries
import os
import time
import random
import csv
from pathlib import Path

# Numerical / ML
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Vision utilities
import torchvision
import torchvision.transforms as transforms
from torchvision import models

# Utilities
from tqdm import tqdm

# Reproducibility check
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: Tesla T4


### Fixed Experimental Configuration (Locked)

This section defines all training hyperparameters and experimental settings.
All values were pre-committed before any training or energy measurements were performed.

These configurations remain fixed for all accuracy-only baseline experiments.


In [2]:
# =========================
# FIXED CONFIGURATION BLOCK
# =========================

CONFIG = {
    # Dataset
    "dataset": "CIFAR-10",
    "num_classes": 10,

    # Models
    "architectures": ["resnet18", "mobilenet_v2"],

    # Training
    "epochs": 100,
    "batch_size": 128,

    # Optimizer
    "optimizer": "SGD",
    "learning_rate": 0.1,
    "momentum": 0.9,
    "weight_decay": 5e-4,

    # LR Scheduler
    "lr_milestones": [50, 75],
    "lr_gamma": 0.1,

    # Reproducibility
    "seeds": [0, 1, 2],

    # Checkpointing
    "checkpoint_dir": "./checkpoints",
    "checkpoint_policy": "best_only",


    # Logging
    "log_dir": "./logs"
}

# Create required directories
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
os.makedirs(CONFIG["log_dir"], exist_ok=True)

# Print config for verification
for k, v in CONFIG.items():
    print(f"{k}: {v}")


dataset: CIFAR-10
num_classes: 10
architectures: ['resnet18', 'mobilenet_v2']
epochs: 100
batch_size: 128
optimizer: SGD
learning_rate: 0.1
momentum: 0.9
weight_decay: 0.0005
lr_milestones: [50, 75]
lr_gamma: 0.1
seeds: [0, 1, 2]
checkpoint_dir: ./checkpoints
checkpoint_policy: best_only
log_dir: ./logs


### Dataset and Data Augmentation

We use the CIFAR-10 dataset with standard preprocessing and data augmentation.
Training data is augmented using random cropping and horizontal flipping.
Validation data uses normalization only.

No adaptive or learned augmentation policies are employed.


In [3]:
# =========================
# DATASET & AUGMENTATION
# =========================

# CIFAR-10 normalization statistics
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

# Training transforms (standard augmentation)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Validation / Test transforms (no augmentation)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Load datasets
train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

# Data loaders
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")


100%|██████████| 170M/170M [00:02<00:00, 74.0MB/s] 


Training samples: 50000
Test samples: 10000


### Model Architectures

We use standard torchvision implementations of ResNet-18 and MobileNet-V2.
Architectures are not modified except for adapting the final classification layer
to match the number of CIFAR-10 classes.

No architectural tuning or efficiency-driven modifications are applied.


In [4]:
# =========================
# MODEL DEFINITIONS
# =========================

def get_model(name, num_classes):
    if name == "resnet18":
        model = models.resnet18(weights=None)
        
        # CIFAR-10 adaptation:
        # Replace first conv (7x7 -> 3x3, stride 1)
        model.conv1 = nn.Conv2d(
            3, 64, kernel_size=3, stride=1, padding=1, bias=False
        )
        model.maxpool = nn.Identity()
        
        # Replace classifier
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features, num_classes
        )

    else:
        raise ValueError(f"Unsupported architecture: {name}")

    return model


# Sanity check: instantiate both models
for arch in CONFIG["architectures"]:
    model = get_model(arch, CONFIG["num_classes"])
    total_params = sum(p.numel() for p in model.parameters())
    print(f"{arch}: {total_params / 1e6:.2f}M parameters")


resnet18: 11.17M parameters
mobilenet_v2: 2.24M parameters


### Optimizer and Learning Rate Scheduler

We optimize all models using stochastic gradient descent (SGD) with momentum.
A fixed multi-step learning rate schedule is applied, with learning rate decays
at epochs 50 and 75.

These settings are fixed across all architectures and runs.


In [5]:
# =========================
# OPTIMIZER & LR SCHEDULER
# =========================

def get_optimizer_and_scheduler(model):
    optimizer = optim.SGD(
        model.parameters(),
        lr=CONFIG["learning_rate"],
        momentum=CONFIG["momentum"],
        weight_decay=CONFIG["weight_decay"]
    )

    scheduler = optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=CONFIG["lr_milestones"],
        gamma=CONFIG["lr_gamma"]
    )

    return optimizer, scheduler


# Sanity check on one model
model = get_model(CONFIG["architectures"][0], CONFIG["num_classes"])
optimizer, scheduler = get_optimizer_and_scheduler(model)

print("Optimizer:", optimizer)
print("Scheduler:", scheduler)


Optimizer: SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    initial_lr: 0.1
    lr: 0.1
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0.0005
)
Scheduler: <torch.optim.lr_scheduler.MultiStepLR object at 0x7e07ab9a9490>


### Energy Logging Integration

We integrate the GPU energy logging pipeline validated in Step-2.
Energy logging runs continuously over the full wall-clock training duration,
including computation, synchronization, idle periods, and checkpointing.

Logging is independent of model training logic.


In [6]:
# =========================
# ENERGY LOGGER (THREADED)
# =========================

import threading

from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetPowerUsage
)

def energy_logging_loop(state):
    while state["running"]:
        ts = time.time()
        power_watts = nvmlDeviceGetPowerUsage(state["handle"]) / 1000.0
        state["writer"].writerow([ts, power_watts])
        time.sleep(state["interval"])

def start_energy_logger(output_csv, interval=0.1):
    nvmlInit()
    handle = nvmlDeviceGetHandleByIndex(0)

    f = open(output_csv, "w", newline="")
    writer = csv.writer(f)
    writer.writerow(["timestamp", "power_watts"])

    state = {
        "handle": handle,
        "file": f,
        "writer": writer,
        "interval": interval,
        "running": True
    }

    thread = threading.Thread(
    target=energy_logging_loop,
    args=(state,),
    daemon=True)
    thread.start()
    
    state["thread"] = thread
    return state

def stop_energy_logger(state):
    state["running"] = False
    state["thread"].join()
    state["file"].close()
    nvmlShutdown()


### Training and Evaluation Loop

Models are trained using a standard supervised learning loop with
cross-entropy loss. Training optimizes accuracy only, without any
energy-aware intervention.

Learning rate scheduling is applied once per epoch.
Validation accuracy is evaluated at the end of each epoch.


In [7]:
# =========================
# TRAINING & EVALUATION
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer):
    model.train()
    correct, total, running_loss = 0, 0, 0.0

    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, targets in loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    acc = 100.0 * correct / total
    return acc


### Checkpointing and Sanity Run

This section integrates training, energy logging, and checkpointing.
A single-epoch sanity run is performed to verify that:

- Training executes end-to-end
- Energy logging spans the full training duration
- Checkpoints are saved correctly

This sanity run does not produce final results.


In [8]:
# =========================
# SANITY RUN (PIPELINE CHECK ONLY)
# =========================

# NOTE:
# This sanity run is used only to verify that the training,
# evaluation, and energy logging pipeline works end-to-end.
# No checkpoints from this run are used or retained.

arch = CONFIG["architectures"][0]   # test with first architecture
seed = CONFIG["seeds"][0]

# Set seeds for reproducibility
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# Model, optimizer, scheduler
model = get_model(arch, CONFIG["num_classes"]).to(device)
optimizer, scheduler = get_optimizer_and_scheduler(model)

# Energy log path (sanity only)
energy_log_path = os.path.join(
    CONFIG["log_dir"],
    f"sanity_energy_{arch}_seed{seed}.csv"
)

# Start energy logging
energy_state = start_energy_logger(energy_log_path)

print("Starting sanity training run (1 epoch)...")
start_time = time.time()

# ---- Train ONE epoch ----
train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
val_acc = evaluate(model, test_loader)
scheduler.step()

# Stop energy logging
stop_energy_logger(energy_state)

elapsed_time = time.time() - start_time

print(f"Sanity run complete for {arch}")
print(f"Train Loss: {train_loss:.4f}")
print(f"Train Acc : {train_acc:.2f}%")
print(f"Val Acc   : {val_acc:.2f}%")
print(f"Elapsed Time: {elapsed_time:.2f} sec")
print(f"Energy log saved to: {energy_log_path}")


Starting sanity training run (1 epoch)...
Sanity run complete for resnet18
Train Loss: 1.9838
Train Acc : 29.49%
Val Acc   : 42.87%
Elapsed Time: 58.51 sec
Energy log saved to: ./logs/sanity_energy_resnet18_seed0.csv


## Step-3.2 — Full Accuracy-Only Baseline Runs

In this step, we execute full accuracy-only training runs using the
pre-committed configuration defined in Step-3.0 and the validated
training pipeline from Step-3.1.

Each run:
- Trains for a fixed 100 epochs
- Optimizes accuracy only
- Logs full wall-clock GPU energy
- Saves model checkpoints
- Uses a fixed random seed

No hyperparameters are tuned during this step.


### Run Configuration: ResNet-18 (Seed 0)

This cell defines the metadata and random seed for a single full
accuracy-only baseline run.

Model: ResNet-18  
Dataset: CIFAR-10  
Seed: 0  
Epochs: 100  

All subsequent cells in this run correspond to this fixed configuration.


### Model, Optimizer, and Scheduler Initialization

This cell instantiates the model, optimizer, and learning rate scheduler
for the current baseline run, using the fixed configuration defined earlier.

No training or energy logging is performed here.


### Energy Logging Start

This cell starts GPU energy logging for the full baseline run.
Energy logging begins immediately before the training loop and will
run continuously until training completes.

This ensures that all training-related GPU activity is captured.


### Full Training Loop (100 Epochs)

This cell executes the full accuracy-only training loop for the current
baseline run. Training proceeds for a fixed 100 epochs with no early stopping.

Validation accuracy is evaluated at the end of each epoch.
Model checkpoints are saved according to the fixed checkpointing policy.
Energy logging remains active throughout the entire training duration.


### Stop Energy Logging and Finalize Run

This cell stops GPU energy logging and records final metadata
for the completed baseline run.

Energy logging is terminated only after all training epochs
have completed, ensuring full wall-clock coverage.


In [9]:
def run_baseline(architecture, seed):
    """
    Runs one full accuracy-only baseline training with energy logging.
    This function is deterministic, disk-safe, and self-contained.
    """

    print("=" * 60)
    print("STARTING BASELINE RUN")
    print(f"Model: {architecture}")
    print(f"Seed : {seed}")
    print("=" * 60)

    # -----------------------
    # Reproducibility
    # -----------------------
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # -----------------------
    # Model & Optimizer
    # -----------------------
    model = get_model(architecture, CONFIG["num_classes"]).to(device)
    optimizer, scheduler = get_optimizer_and_scheduler(model)

    # -----------------------
    # Energy Logging
    # -----------------------
    energy_log_file = os.path.join(
        CONFIG["log_dir"],
        f"energy_{architecture}_seed{seed}.csv"
    )

    energy_state = start_energy_logger(
        output_csv=energy_log_file,
        interval=0.1  # locked sampling rate
    )

    start_time = time.time()
    best_val_acc = 0.0

    # -----------------------
    # Training Loop
    # -----------------------
    for epoch in range(1, CONFIG["epochs"] + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer
        )
        val_acc = evaluate(model, test_loader)
        scheduler.step()

        # BEST-only checkpointing
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            ckpt_path = os.path.join(
                CONFIG["checkpoint_dir"],
                f"{architecture}_seed{seed}_BEST.pt"
            )
            torch.save(
                {
                    "epoch": epoch,
                    "val_acc": val_acc,
                    "model_state_dict": model.state_dict(),
                },
                ckpt_path
            )

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"Epoch [{epoch:03d}/{CONFIG['epochs']}]: "
                f"Train Acc={train_acc:.2f}%, "
                f"Val Acc={val_acc:.2f}%, "
                f"LR={scheduler.get_last_lr()[0]:.4f}"
            )

    # -----------------------
    # Stop Energy Logging
    # -----------------------
    stop_energy_logger(energy_state)

    total_time = time.time() - start_time

    print("=" * 60)
    print("RUN COMPLETE")
    print(f"Model             : {architecture}")
    print(f"Seed              : {seed}")
    print(f"Best Val Accuracy : {best_val_acc:.2f}%")
    print(f"Training Time (s) : {total_time:.2f}")
    print(f"Energy CSV        : {energy_log_file}")
    print("=" * 60)

    return {
        "architecture": architecture,
        "seed": seed,
        "best_val_acc": best_val_acc,
        "training_time_sec": total_time,
        "energy_csv": energy_log_file
    }


In [10]:
result_1 = run_baseline("resnet18", 0)


STARTING BASELINE RUN
Model: resnet18
Seed : 0
Epoch [001/100]: Train Acc=30.79%, Val Acc=42.52%, LR=0.1000
Epoch [005/100]: Train Acc=73.08%, Val Acc=69.14%, LR=0.1000
Epoch [010/100]: Train Acc=82.94%, Val Acc=81.32%, LR=0.1000
Epoch [015/100]: Train Acc=85.33%, Val Acc=74.03%, LR=0.1000
Epoch [020/100]: Train Acc=86.62%, Val Acc=78.89%, LR=0.1000
Epoch [025/100]: Train Acc=87.53%, Val Acc=80.67%, LR=0.1000
Epoch [030/100]: Train Acc=87.84%, Val Acc=83.32%, LR=0.1000
Epoch [035/100]: Train Acc=88.19%, Val Acc=83.10%, LR=0.1000
Epoch [040/100]: Train Acc=88.66%, Val Acc=81.10%, LR=0.1000
Epoch [045/100]: Train Acc=88.75%, Val Acc=83.72%, LR=0.1000
Epoch [050/100]: Train Acc=88.82%, Val Acc=84.06%, LR=0.0100
Epoch [055/100]: Train Acc=97.15%, Val Acc=93.71%, LR=0.0100
Epoch [060/100]: Train Acc=98.29%, Val Acc=93.88%, LR=0.0100
Epoch [065/100]: Train Acc=98.67%, Val Acc=93.74%, LR=0.0100
Epoch [070/100]: Train Acc=98.76%, Val Acc=93.23%, LR=0.0100
Epoch [075/100]: Train Acc=98.90%, Val

In [11]:
!wc -l ./logs/energy_resnet18_seed0.csv


57012 ./logs/energy_resnet18_seed0.csv


In [12]:
result_2 = run_baseline("resnet18", 1)


STARTING BASELINE RUN
Model: resnet18
Seed : 1
Epoch [001/100]: Train Acc=28.85%, Val Acc=43.75%, LR=0.1000
Epoch [005/100]: Train Acc=73.17%, Val Acc=70.31%, LR=0.1000
Epoch [010/100]: Train Acc=82.73%, Val Acc=79.14%, LR=0.1000
Epoch [015/100]: Train Acc=85.00%, Val Acc=81.08%, LR=0.1000
Epoch [020/100]: Train Acc=86.60%, Val Acc=76.15%, LR=0.1000
Epoch [025/100]: Train Acc=87.36%, Val Acc=83.72%, LR=0.1000
Epoch [030/100]: Train Acc=87.78%, Val Acc=83.45%, LR=0.1000
Epoch [035/100]: Train Acc=88.06%, Val Acc=80.67%, LR=0.1000
Epoch [040/100]: Train Acc=88.53%, Val Acc=86.04%, LR=0.1000
Epoch [045/100]: Train Acc=88.45%, Val Acc=84.30%, LR=0.1000
Epoch [050/100]: Train Acc=88.85%, Val Acc=86.02%, LR=0.0100
Epoch [055/100]: Train Acc=97.18%, Val Acc=93.06%, LR=0.0100
Epoch [060/100]: Train Acc=98.29%, Val Acc=93.35%, LR=0.0100
Epoch [065/100]: Train Acc=98.75%, Val Acc=93.12%, LR=0.0100
Epoch [070/100]: Train Acc=98.85%, Val Acc=93.63%, LR=0.0100
Epoch [075/100]: Train Acc=98.83%, Val

In [13]:
!wc -l ./logs/energy_resnet18_seed1.csv


56886 ./logs/energy_resnet18_seed1.csv


In [14]:
result_3 = run_baseline("resnet18", 2)


STARTING BASELINE RUN
Model: resnet18
Seed : 2
Epoch [001/100]: Train Acc=25.13%, Val Acc=40.24%, LR=0.1000
Epoch [005/100]: Train Acc=71.79%, Val Acc=73.82%, LR=0.1000
Epoch [010/100]: Train Acc=82.47%, Val Acc=80.70%, LR=0.1000
Epoch [015/100]: Train Acc=85.32%, Val Acc=82.33%, LR=0.1000
Epoch [020/100]: Train Acc=86.59%, Val Acc=83.61%, LR=0.1000
Epoch [025/100]: Train Acc=87.52%, Val Acc=83.51%, LR=0.1000
Epoch [030/100]: Train Acc=87.89%, Val Acc=83.92%, LR=0.1000
Epoch [035/100]: Train Acc=88.20%, Val Acc=84.10%, LR=0.1000
Epoch [040/100]: Train Acc=88.42%, Val Acc=84.37%, LR=0.1000
Epoch [045/100]: Train Acc=88.87%, Val Acc=85.55%, LR=0.1000
Epoch [050/100]: Train Acc=89.03%, Val Acc=80.14%, LR=0.0100
Epoch [055/100]: Train Acc=97.27%, Val Acc=93.45%, LR=0.0100
Epoch [060/100]: Train Acc=98.39%, Val Acc=93.45%, LR=0.0100
Epoch [065/100]: Train Acc=98.67%, Val Acc=93.41%, LR=0.0100
Epoch [070/100]: Train Acc=98.85%, Val Acc=93.33%, LR=0.0100
Epoch [075/100]: Train Acc=98.85%, Val

In [15]:
!wc -l ./logs/energy_resnet18_seed2.csv


57281 ./logs/energy_resnet18_seed2.csv


In [ ]:
result_4 = run_baseline("mobilenet_v2", 0)


STARTING BASELINE RUN
Model: mobilenet_v2
Seed : 0
Epoch [001/100]: Train Acc=14.17%, Val Acc=18.30%, LR=0.1000
Epoch [005/100]: Train Acc=35.73%, Val Acc=36.14%, LR=0.1000
Epoch [010/100]: Train Acc=50.34%, Val Acc=51.67%, LR=0.1000
Epoch [015/100]: Train Acc=58.38%, Val Acc=56.86%, LR=0.1000
Epoch [020/100]: Train Acc=61.68%, Val Acc=61.45%, LR=0.1000
Epoch [025/100]: Train Acc=63.63%, Val Acc=61.37%, LR=0.1000
Epoch [030/100]: Train Acc=64.07%, Val Acc=54.62%, LR=0.1000
Epoch [035/100]: Train Acc=65.11%, Val Acc=58.27%, LR=0.1000
Epoch [040/100]: Train Acc=66.79%, Val Acc=64.20%, LR=0.1000
Epoch [045/100]: Train Acc=67.85%, Val Acc=67.72%, LR=0.1000
Epoch [050/100]: Train Acc=68.03%, Val Acc=65.01%, LR=0.0100
Epoch [055/100]: Train Acc=78.56%, Val Acc=79.67%, LR=0.0100
Epoch [060/100]: Train Acc=79.85%, Val Acc=78.71%, LR=0.0100
Epoch [065/100]: Train Acc=80.25%, Val Acc=80.43%, LR=0.0100
Epoch [070/100]: Train Acc=80.41%, Val Acc=80.82%, LR=0.0100


In [ ]:
!wc -l ./logs/energy_mobilenet_v2_seed0.csv


In [ ]:
result_5 = run_baseline("mobilenet_v2", 1)


In [ ]:
!wc -l ./logs/energy_mobilenet_v2_seed1.csv


In [ ]:
result_6 = run_baseline("mobilenet_v2", 2)


In [ ]:
!wc -l ./logs/energy_mobilenet_v2_seed2.csv


In [ ]:
import pandas as pd
import numpy as np
import os

# =========================
# ENERGY INTEGRATION HELPER
# =========================
def compute_energy_joules(csv_path):
    """
    Computes total energy (Joules) by numerically integrating
    GPU power over wall-clock time using the trapezoidal rule.
    """
    df = pd.read_csv(csv_path)
    timestamps = df["timestamp"].values
    power = df["power_watts"].values

    energy_joules = np.trapezoid(power, timestamps)
    return energy_joules


# =========================
# COLLECT RUN RESULTS
# =========================
ALL_RESULTS = [
    result_1, result_2, result_3,
    result_4, result_5, result_6
]

summary = {}

for res in ALL_RESULTS:
    model = res["architecture"]

    if model not in summary:
        summary[model] = {
            "accuracy": [],
            "time": [],
            "energy": []
        }

    # Accuracy
    summary[model]["accuracy"].append(res["best_val_acc"])

    # Training time
    summary[model]["time"].append(res["training_time_sec"])

    # Energy
    energy_j = compute_energy_joules(res["energy_csv"])
    summary[model]["energy"].append(energy_j)


# =========================
# PRINT PHASE-1 SUMMARY
# =========================
print("\n" + "=" * 90)
print("PHASE-1 SUMMARY — ACCURACY-ONLY BASELINES (REPRODUCIBLE)")
print("=" * 90)

print(f"{'Model':15s} | {'Acc (%)':17s} | {'Time (s)':17s} | {'Energy (kJ)':17s}")
print("-" * 90)

for model, vals in summary.items():
    acc = np.array(vals["accuracy"])
    time_s = np.array(vals["time"])
    energy_kj = np.array(vals["energy"]) / 1000.0

    print(
        f"{model:15s} | "
        f"{acc.mean():.2f} ± {acc.std():.2f} | "
        f"{time_s.mean():.1f} ± {time_s.std():.1f} | "
        f"{energy_kj.mean():.2f} ± {energy_kj.std():.2f}"
    )

print("=" * 90)
